# RAG Practice

## Chunk Text

In [1]:
text = open("cricket_chapter.txt").read()


def make_chunks(text, chunk_size=50, overlap=10):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = start + chunk_size - overlap
    return chunks


chunks = make_chunks(text)
print(len(chunks))
print(chunks[0])

59
Cricket: A Complete Introduction Origins and history. Cricket is a bat and ball sport that began in south-east England, where it was played in villages and open fields for hundreds of years before it became organised. The rules, called the Laws of Cricket, are written and looked after by the


## TF-IDF Search

In [2]:
import math
import re


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def build_tfidf(chunks):
    docs = []
    for c in chunks:
        docs.append(tokenize(c))
    n = len(docs)

    df = {}
    for doc in docs:
        for word in set(doc):
            if word in df:
                df[word] = df[word] + 1
            else:
                df[word] = 1

    idf = {}
    for word in df:
        idf[word] = math.log(n / df[word]) + 1

    vectors = []
    for doc in docs:
        vec = {}
        for word in doc:
            if word in vec:
                vec[word] = vec[word] + 1
            else:
                vec[word] = 1
        for word in vec:
            vec[word] = (vec[word] / len(doc)) * idf[word]
        vectors.append(vec)
    return vectors, idf


def cosine(a, b):
    dot = 0
    for w in a:
        if w in b:
            dot = dot + a[w] * b[w]

    sum_a = 0
    for v in a.values():
        sum_a = sum_a + v * v
    sum_b = 0
    for v in b.values():
        sum_b = sum_b + v * v
    na = math.sqrt(sum_a)
    nb = math.sqrt(sum_b)
    if na == 0 or nb == 0:
        return 0
    return dot / (na * nb)


def search(query, chunks, vectors, idf, top_k=3):
    words = tokenize(query)
    q = {}
    for w in words:
        if w in idf:
            if w in q:
                q[w] = q[w] + 1
            else:
                q[w] = 1
    for w in q:
        q[w] = (q[w] / len(words)) * idf[w]

    scores = []
    for i in range(len(vectors)):
        s = cosine(q, vectors[i])
        scores.append((s, i))
    scores.sort(reverse=True)

    results = []
    for s, i in scores[:top_k]:
        results.append((s, chunks[i]))
    return results


vectors, idf = build_tfidf(chunks)
for score, chunk in search("what is a yorker?", chunks, vectors, idf):
    print(round(score, 3), chunk[:100])

0.29 reverse swing may begin. Tampering with the ball by using any substance is illegal. A yorker is a ba
0.101 the match is a tie, while a Test match that runs out of time without a winner is called a draw. The 
0.097 the head or chest of the batter. A full toss reaches the batter without bouncing at all. A beamer is


## Install Embeddings

In [3]:
!pip install -q sentence-transformers

## Embedding Search

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = model.encode(chunks)


def embedding_search(query, chunks, chunk_embeddings, top_k=3):
    q = model.encode([query])[0]
    scores = []
    for i in range(len(chunk_embeddings)):
        v = chunk_embeddings[i]
        s = np.dot(q, v) / (np.linalg.norm(q) * np.linalg.norm(v))
        scores.append((s, i))
    scores.sort(reverse=True)
    results = []
    for s, i in scores[:top_k]:
        results.append((float(s), chunks[i]))
    return results


for score, chunk in embedding_search("what is a yorker?", chunks, chunk_embeddings):
    print(round(score, 3), chunk[:100])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.343 reverse swing may begin. Tampering with the ball by using any substance is illegal. A yorker is a ba
0.271 a delivery with hidden lower speed that is meant to deceive the batter. Batting. A batter scores run
0.222 has eleven players, including the bowler and the wicketkeeper. The other fielders stand in different


## Hybrid Search

In [5]:
def tfidf_scores(query, vectors, idf):
    words = tokenize(query)
    q = {}
    for w in words:
        if w in idf:
            q[w] = q.get(w, 0) + 1
    for w in q:
        q[w] = (q[w] / len(words)) * idf[w]
    scores = []
    for v in vectors:
        scores.append(cosine(q, v))
    return scores


def embedding_scores(query, chunk_embeddings):
    q = model.encode([query])[0]
    scores = []
    for v in chunk_embeddings:
        s = np.dot(q, v) / (np.linalg.norm(q) * np.linalg.norm(v))
        scores.append(float(s))
    return scores


def normalize(scores):
    low = min(scores)
    high = max(scores)
    result = []
    for s in scores:
        if high == low:
            result.append(0)
        else:
            result.append((s - low) / (high - low))
    return result


def hybrid_search(query, chunks, vectors, idf, chunk_embeddings, weight=0.5, top_k=3):
    a = normalize(tfidf_scores(query, vectors, idf))
    b = normalize(embedding_scores(query, chunk_embeddings))
    combined = []
    for i in range(len(chunks)):
        score = weight * a[i] + (1 - weight) * b[i]
        combined.append((score, i))
    combined.sort(reverse=True)
    results = []
    for score, i in combined[:top_k]:
        results.append((score, chunks[i]))
    return results


q = "what is a yorker?"
for score, chunk in hybrid_search(q, chunks, vectors, idf, chunk_embeddings):
    print(round(score, 3), chunk[:100])

1.0 reverse swing may begin. Tampering with the ball by using any substance is illegal. A yorker is a ba
0.489 a delivery with hidden lower speed that is meant to deceive the batter. Batting. A batter scores run
0.452 the head or chest of the batter. A full toss reaches the batter without bouncing at all. A beamer is


## Query Check

In [6]:
for q in ["what is run out", "what is run out?"]:
    top = search(q, chunks, vectors, idf, top_k=1)
    print(q, "->", round(top[0][0], 3), top[0][1][:60])

what is run out -> 0.318 batter can only be run out. A bye is a run taken when the ba
what is run out? -> 0.318 batter can only be run out. A bye is a run taken when the ba


## Build Prompt

In [7]:
def build_prompt(question, top_chunks):
    context = ""
    for score, chunk in top_chunks:
        context = context + chunk + "\n\n"
    prompt = (
        "Answer the question using only the notes below. "
        "If the answer is not in the notes, say you don't know.\n\n"
        "Notes:\n" + context +
        "Question: " + question + "\nAnswer:"
    )
    return prompt

## Evaluation

In [8]:
from test_set import test_set

for question, key in test_set:
    count = 0
    for c in chunks:
        if key.lower() in c.lower():
            count = count + 1
    if count != 1:
        print("CHECK THIS KEY:", key, "appears in", count, "chunks")


def run_tfidf(q):
    return search(q, chunks, vectors, idf, top_k=3)


def run_embedding(q):
    return embedding_search(q, chunks, chunk_embeddings, top_k=3)


def run_hybrid(q):
    return hybrid_search(q, chunks, vectors, idf, chunk_embeddings, weight=0.5, top_k=3)


def evaluate(search_function, test_set, show=True):
    top1 = 0
    top3 = 0
    for question, key in test_set:
        results = search_function(question)
        found_at = None
        for position in range(len(results)):
            if key.lower() in results[position][1].lower():
                found_at = position
                break
        if found_at == 0:
            top1 = top1 + 1
        elif show:
            print("   missed top-1:", question)
        if found_at is not None:
            top3 = top3 + 1
    total = len(test_set)
    return top1 / total, top3 / total


for name, fn in [("TF-IDF", run_tfidf), ("Embedding", run_embedding), ("Hybrid", run_hybrid)]:
    print(name)
    t1, t3 = evaluate(fn, test_set)
    print("   top-1:", round(t1 * 100), "%   top-3:", round(t3 * 100), "%")

TF-IDF
   missed top-1: how long is a cricket pitch?
   missed top-1: how many stumps are in a wicket?
   missed top-1: what is an over?
   missed top-1: what does LBW stand for?
   missed top-1: who looks after the laws of cricket?
   missed top-1: how many overs are in a T20 innings?
   missed top-1: how many overs can one bowler bowl in an ODI?
   missed top-1: what is the Ashes?
   missed top-1: what is a cricket ball made of?
   missed top-1: wat is a no ball
   missed top-1: how many reviews in drs
   missed top-1: economy rate of bowler
   missed top-1: stumped meaning?
   missed top-1: who decides the batting order and where fielders stand
   missed top-1: how wide can the bat be
   missed top-1: what happens if a bowler bends the arm too much
   missed top-1: how fast can pace bowlers bowl
   top-1: 66 %   top-3: 94 %
Embedding
   missed top-1: how many stumps are in a wicket?
   missed top-1: what does LBW stand for?
   missed top-1: what is a free hit?
   missed top-1: how m

## Weight Sweep

In [9]:
for w in [0, 0.2, 0.4, 0.6, 0.8, 1]:
    def run_this(q):
        return hybrid_search(q, chunks, vectors, idf, chunk_embeddings, weight=w, top_k=3)
    t1, t3 = evaluate(run_this, test_set, show=False)
    print("weight", w, "top-1:", round(t1 * 100), "%  top-3:", round(t3 * 100), "%")

weight 0 top-1: 68 %  top-3: 94 %
weight 0.2 top-1: 84 %  top-3: 98 %
weight 0.4 top-1: 80 %  top-3: 100 %
weight 0.6 top-1: 74 %  top-3: 100 %
weight 0.8 top-1: 72 %  top-3: 98 %
weight 1 top-1: 66 %  top-3: 94 %


## Install Groq

In [10]:
!pip install -q groq

## Connect Groq

In [11]:
from groq import Groq
import getpass

api_key = getpass.getpass("Paste your Groq key: ").strip()
if api_key == "":
    raise ValueError("The key is empty. Run this cell again and paste your key.")
client = Groq(api_key=api_key)

Paste your Groq key: ··········


## Chat Models

In [12]:
skip = ["whisper", "tts", "guard", "orpheus", "allam"]
for m in client.models.list().data:
    ok = True
    for word in skip:
        if word in m.id:
            ok = False
    if ok:
        print(m.id)

openai/gpt-oss-20b
qwen/qwen3.8-27b
openai/gpt-oss-120b


## Ask LLM

In [15]:
model_name = "openai/gpt-oss-20b"


def ask_llm(prompt):
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content


def answer_question(question):
    top_chunks = hybrid_search(question, chunks, vectors, idf, chunk_embeddings, weight=0.4, top_k=3)
    prompt = build_prompt(question, top_chunks)
    return ask_llm(prompt)

## Test Answers

In [16]:
print(answer_question("What is run out?"))
print()
print(answer_question("What is a yorker?"))
print()
print(answer_question("Who won the 2011 World Cup?"))

A run out is a dismissal that occurs when a fielder (or bowler) dislodges the wicket with the ball while the batter or non‑striker is out of their crease. Since 2022, a bowler who runs out a non‑striker for leaving the crease early is treated as a normal run out.

A yorker is a ball pitched right at the batter's feet and is very hard to hit.

I don't know.
